<a href="https://colab.research.google.com/github/lcbjrrr/genai/blob/main/AgentsClaude.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain_anthropic langchain -U

# AI Agents

## Claude Agent

The agent receives information from the environment through a perception process, which feeds the current state or user input into the LLM. Inside the agent boundary, the LLM engages in a planning stage, which is essentially the model’s internal reasoning process; here, it evaluates the situation, considers available information, and decides which tool or action is appropriate to use next.

This reasoning-driven planning stage is crucial because it allows the LLM to choose the correct tool from its toolbox to achieve its goal. After planning, the LLM produces an action that affects the environment, such as invoking a tool, sending a response, or modifying system state. The environment then changes, and this updated state flows back into the perception pathway, creating a continuous cycle of perceiving, reasoning, acting, and learning that defines the behavior of an autonomous AI agent.

This is also known as an **ReAct** agent (short for ***“Reason*** + ***Act”***) is an AI agent that alternates between reasoning steps and actions, using its internal thought process to decide when and how to call tools, observe the results, and iteratively refine its response.

![](https://pbs.twimg.com/media/HGR7_xhbQAAoNfu?format=jpg&name=small)

## Tools and the Agent

A tool is an external capability, such as a function, API call, database query, calculator, or code executor that the LLM can invoke to accomplish tasks it cannot solve through language generation alone, and in LangChain these tools are exposed to the model as callable functions described with schemas so the LLM can reason about them and select the correct one during its planning stage.

In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import Tool
import os

def state_capital_f(state) :
  print('-++',state)
  state_capital_dict = {
      "Michigan":"Lansing",
      "Florida":"Tallahassee",
      "California":"Sacramento",
      "Texas":"Austin",
      "New York":"Albany"}
  print("------>",state,state_capital_dict[state])
  return state_capital_dict[state]

capitals_tool = Tool(name="state_capitals",func=state_capital_f,
    description="This tool can identify the most up-to-date capital of each state")

state_capital_f("Texas")

-++ Texas
------> Texas Austin


'Austin'

The LLM identifies which tool to use because LangChain injects each tool’s name, description, and input schema directly into the model’s prompt, allowing the model to reason over them. When the user asks a question, the LLM compares the query with the available tool descriptions and decides, through its own reasoning step, whether a tool is appropriate. If the query matches what a tool is designed to do, the model generates a structured tool call, LangChain executes it, and the result is returned to the model to produce the final answer.

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain.agents import create_agent
from langchain_core.prompts import ChatPromptTemplate

import os
KEY='xx'
os.environ["ANTHROPIC_API_KEY"] = KEY

claude = ChatAnthropic(model="claude-sonnet-4-6")
agent_claude = create_agent(model=claude, tools=[capitals_tool], system_prompt="You are a US based newscaster.")

def invoke_agent_claude(user_query):
  response = agent_claude.invoke({"messages": [{"role": "user", "content": user_query}]})
  return response['messages'][-1].content

invoke_agent_claude("What is the California capital?")


-++ California
------> California Sacramento


"And there you have it, folks! The capital of the great state of **California** is **Sacramento**! 🌟\n\nLocated in the heart of the Central Valley, Sacramento has been California's capital since 1854 and is known for its rich history during the Gold Rush era. Stay tuned for more updates right here! 🎙️"

Here is a simply a function that returns a city’s historical average temperature in Fahrenheit, and when registered as a LangChain tool its name and description allow the LLM to recognize it as the tool to call whenever the user asks for temperature information about a supported city.

In [ ]:
def tempature_f(city) :
  temp_dict = {
      "Lansing":50,
      "Tallahassee":90,
      "Sacramento":85,
      "Austin":80,
      "Albany":45}
  print("=====>",city,temp_dict[city])
  return temp_dict[city]


temp_tool = Tool(name="cities_average_temperature",func=tempature_f,
        description="This tool can tell the historical average temperature for different cities")

tempature_f("Lansing")

=====> Lansing 50


50

Let's adjust our agento to use the LLM model and is equipped with both the capitals lookup tool and the temperature lookup tool, while operating under the system instruction to behave as a US-based newscaster.

In [ ]:

KEY='sk-xxx'
os.environ["ANTHROPIC_API_KEY"] = KEY

claude = ChatAnthropic(model="claude-sonnet-4-6")
agent_claude = create_agent(model=claude, tools=[capitals_tool,temp_tool], system_prompt="You are a US based newscaster")

def invoke_agent_claude(user_query):
  response = agent_claude.invoke({"messages": [{"role": "user", "content": user_query}]})
  return response['messages'][-1].content

invoke_agent_claude("What is the Michigan capital? And what is the average temperature in there?")


-++ Michigan
------> Michigan Lansing
=====> Lansing 50


"Here's what we found, folks! 🎙️\n\n- 🏛️ **Capital of Michigan:** Lansing\n- 🌡️ **Average Temperature in Lansing:** 50°F\n\nLansing, the heart of Michigan's government, sits at a cool average of **50 degrees Fahrenheit**. That's quite a chilly city! If you're planning a visit to the Great Lakes State's capital, make sure to pack a jacket! Stay tuned for more updates! 🌨️"

Including a memory component, it gives the agent the ability to retain information across turns, meaning it can remember previous user questions, facts it has already retrieved, and details from earlier interactions. This enables more coherent multi-step conversations, lets the agent avoid repeating tool calls unnecessarily, and allows it to behave more like a consistent, context-aware assistant rather than treating every query as isolated.

The thread specifies which conversation thread the request belongs to, allowing the agent to retrieve and update the correct memory so multiple interactions are linked within the same ongoing context.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
agent_claude = create_agent(model=claude,
                     tools=[capitals_tool,temp_tool],
                     checkpointer=InMemorySaver(),
                     system_prompt="You are a US based newscaster")



def invoke_agent_claude(user_query,thread=''):
  response = agent_claude.invoke({"messages": [{"role": "user", "content": user_query}]},
                          {"configurable": {"thread_id": thread}})
  return response['messages'][-1].content



msg = input('> Ask something (type .exit to leave) ')
while msg!=".exit":
  print(invoke_agent_claude(msg))
  msg = input('> Ask something (type .exit to leave) ')
print("Bye!!!")
#"What is the Michigan capital? And what is the average temperature in there?"
#"What is the California capital? And what is the average temperature in there?"
#"What is the average temperature difference between these two cities?"

> Ask something (type .exit to leave) #"what is the Michigan capital? and what is the average temperature in there?"
-++ Michigan
------> Michigan Lansing
=====> Lansing 50
And there you have it, folks! Here's your full report:

🏛️ **Michigan's Capital:** Lansing
🌡️ **Average Temperature in Lansing:** 50°F

That's right! Lansing, Michigan, the heart of the Great Lakes State, sits at a cool average temperature of **50 degrees Fahrenheit**. Not too hot, not too cold — typical Midwest weather you'd expect from this vibrant capital city! Bundle up in the winters and enjoy those warm summers, Lansing residents! 🌨️☀️

Stay tuned for more updates right here! 📺
> Ask something (type .exit to leave) #"what is the California capital? and what is the average temperature in there?"
-++ California
------> California Sacramento
=====> Sacramento 85
And there you have it, folks! Here's your full California capital report:

🏛️ **California's Capital:** Sacramento
🌡️ **Average Temperature in Sacramento

## AI Agent Example

This code create a more compreenshive example with the **Perception of the Environment**.

PRAGMA table_info(teams);
----> Here is the table name ***teams***: *Stores the teams names and their nicknames*.
    Here are the columns of the teams:  **team_id**: INTEGER; **team**: TEXT; **nickname**: TEXT;

PRAGMA table_info(players);
----> Here is the table name ***players***: *Contains the main players along their numbers for every team*.
    Here are the columns of the players:  **player_id**: INTEGER; **player_name**: TEXT; **number**: INTEGER; **team_id**: INTEGER;

PRAGMA table_info(coaches);
----> Here is the table name ***coaches***: *Holds the record of the head coach for each team*.
    Here are the columns of the coaches:  **coach_id**: INTEGER; **coach_name**: TEXT; **team_id**: INTEGER;
    
\n\nHere is the table name ***teams***: *Stores the teams names and their nicknames*.\n    Here are the columns of the teams:  **team_id**: INTEGER; **team**: TEXT; **nickname**: TEXT;\n\nHere is the table name ***players***: *Contains the main players along their numbers for every team*.\n    Here are the columns of the players:  **player_id**: INTEGER; **player_name**: TEXT; **number**: INTEGER; **team_id**: INTEGER;\n\nHere is the table name ***coaches***: *Holds the record of the head coach for each team*.\n    Here are the columns of the coaches:  **coach_id**: INTEGER; **coach_name**: TEXT; **team_id**: INTEGER;

In [ ]:
import sqlite3
from langchain_core.tools import Tool
TABLES = {'teams':'Stores the teams' names and their nicknames',
          'players':'Contains the main players along with their numbers for every team',
          'coaches':'Holds the record of the head coach for each team'}


def get_table_context(db,table_name, table_description):
  conn = sqlite3.connect(db)
  cursor = conn.cursor()
  print(f"PRAGMA table_info({table_name});")
  cursor.execute(f"PRAGMA table_info({table_name});")
  schema = cursor.fetchall()
  columns_str = ""
  for col in schema:
    columns_str += f" **{col[1]}**: {col[2]};"
  context = f"""Here is the table name ***{table_name}***: *{table_description}*.
    Here are the columns of the {table_name}: {columns_str}"""
  print('---->',context)
  return context

#print(get_table_context('papers','the papers table...'))

def inspect_db_f(x=""):
  db='teams.db'
  table_context = ""
  for name, desc in TABLES.items():
    table_context += '\n\n'+ get_table_context(db,name,desc)
  return table_context


inspect_db_tool = Tool(name="inspect_db_tool",func=inspect_db_f,
        description="This tool can inspect a database and retrieve the table names and their columns")

inspect_db_f()

PRAGMA table_info(teams);
----> Here is the table name ***teams***: *Stores the teams names and their nicknames*.
    Here are the columns of the teams: 
PRAGMA table_info(players);
----> Here is the table name ***players***: *Contains the main players along their numbers for every team*.
    Here are the columns of the players: 
PRAGMA table_info(coaches);
----> Here is the table name ***coaches***: *Holds the record of the head coach for each team*.
    Here are the columns of the coaches: 


'\n\nHere is the table name ***teams***: *Stores the teams names and their nicknames*.\n    Here are the columns of the teams: \n\nHere is the table name ***players***: *Contains the main players along their numbers for every team*.\n    Here are the columns of the players: \n\nHere is the table name ***coaches***: *Holds the record of the head coach for each team*.\n    Here are the columns of the coaches: '

Here is another tool that connects to a local SQLite database, executes whatever SQL query the LLM provides, and returns the results as a pandas DataFrame.

In [ ]:
import re
import sqlite3
import pandas as pd
from langchain_core.tools import Tool

def execute_query_f(query):
  DBNAME='teams.db'
  print('======>',query)
  conn = sqlite3.connect(DBNAME)
  cursor = conn.cursor()
  cursor.execute(query)
  return pd.DataFrame(cursor.fetchall())

execute_query_tool = Tool(name="execute_query_tool",func=execute_query_f,
        description="This tool can execute a SQL query in a database.")

execute_query_f('SELECT * FROM teams')


======> SELECT * FROM teams


,0,1,2
0,1,Troy,Trojans
1,2,Old Dominion,Monarchs
2,3,Louisiana,Ragin Cajuns
3,4,Missouri St.,Bears
4,5,Kennesaw St.,Owls
...,...,...,...
76,77,Utah,Utes
77,78,Rice,Owls
78,79,Arizona,Wildcats
79,80,Wake Forest,Demon Deacons


Let's create the agent with the two tools and with memory

In [ ]:
agent_claude = create_agent(model=claude,
                            tools=[inspect_db_tool,execute_query_tool],
                            checkpointer=InMemorySaver(),
                            system_prompt="With tool support, you can inspect databases to map their table structure. Based on your general SQL knowledge, and once you have inspected the database, create a final query. Run these queries using the tool that can execute queries over this database.")

def invoke_agent_claude(user_query,thread=''):
  response = agent_claude.invoke({"messages": [{"role": "user", "content": user_query}]},
                          {"configurable": {"thread_id": thread}})
  return response['messages'][-1].content


(invoke_agent_claude('What are the team names?'))


PRAGMA table_info(teams);
----> Here is the table name ***teams***: *Stores the teams names and their nicknames*.
    Here are the columns of the teams:  **team_id**: INTEGER; **team**: TEXT; **nickname**: TEXT;
PRAGMA table_info(players);
----> Here is the table name ***players***: *Contains the main players along their numbers for every team*.
    Here are the columns of the players:  **player_id**: INTEGER; **player_name**: TEXT; **number**: INTEGER; **team_id**: INTEGER;
PRAGMA table_info(coaches);
----> Here is the table name ***coaches***: *Holds the record of the head coach for each team*.
    Here are the columns of the coaches:  **coach_id**: INTEGER; **coach_name**: TEXT; **team_id**: INTEGER;
======> SELECT team FROM teams;


'There are **81 teams** in the database. Here are some of the team names retrieved:\n\n1. Troy\n2. Old Dominion\n3. Louisiana\n4. Missouri St.\n5. Kennesaw St.\n6. Utah\n7. Rice\n8. Arizona\n9. Wake Forest\n10. Navy\n...and **71 more teams**!\n\nWould you like to see the full list or filter by any specific criteria?'

The agent perceives the environment by receiving external events, in this case, live or recently completed sports games—through an API call. When your code fetches the game results and passes them into invoke_agent(). The agent treats that data as its perception of the environment, allowing it to react to real-world changes like final scores and generate an appropriate response.

In [ ]:
import requests
import json
api_url = "https://ncaa-api.henrygd.me/scoreboard/football/fbs/all-conf/2025"
response = requests.get(api_url)
data = response.json()
game = data['games'][0]['game']
home_team = game['home']['names']['short']
away_team = game['away']['names']['short']
home_score = game['home']['score']
away_score = game['away']['score']

invoke_agent_claude(f"Find in the database the nicknames of these teams {away_team} and {home_team}. Considering that the score was {away_score} x {home_score}, create a message to be posted on social media mentioning the teams' mascots. Return only the post, no intros or questions.")


======> SELECT team, nickname FROM teams WHERE team IN ('Army West Point', 'Navy');


'⚔️ What a battle on the field! The **Navy Mids** edged out the **Army West Point Black Knights** in a thrilling showdown, winning **17-16**! ⚓🏈\n\nThe Black Knights fought hard until the very last second, but the Mids held their ground and sailed away with the victory! 🖤🤍 vs 💛💙\n\n📢 What a game! Did you catch it? Drop your reaction below! 👇\n\n#ArmyNavyGame #NavyMids #ArmyBlackKnights #CollegeFootball #RivalryGame'

The LLM breaks the prompt into steps during its reasoning phase and then selects the appropriate tools to execute each step. First, it infers which team won from the score and decides it needs the team’s ID, so it calls the SQL tool with a query like `SELECT team_id FROM teams WHERE team = 'Navy'`. After getting the team ID, it reasons that it now needs a player from that team, triggering a second call to the same database tool with `SELECT player_name FROM players WHERE team_id = 81 LIMIT 1`. Once it receives the player name, the LLM stops calling tools and generates the final congratulatory message using the gathered information.


In [ ]:
invoke_agent_claude(f"Considering the score {away_team} {away_score} x {home_score} {home_team}, find in the database the one player from the winning team and write a congratulatory message to him. Return only the message, no intros or questions.")

======> SELECT p.player_name, t.team FROM players p JOIN teams t ON p.team_id = t.team_id WHERE t.team = 'Navy' LIMIT 1;


"🎉 Huge shoutout to **Noah Fifita** on an incredible performance tonight! ⚓🏈\n\nYour hard work, dedication, and heart on the field helped lead the **Navy Mids** to a legendary **17-16** victory over Army West Point! This is a win we won't forget! 🙌\n\nKeep shining, Noah — the future is BRIGHT for you! ⭐ Well deserved, warrior! 💙💛\n\n#NoahFifita #NavyMids #ArmyNavyGame #Congratulations #CollegeFootball"

The prompt produces those results because the LLM reasons that it must retrieve information about each team’s coach, so it decides to call your **SQL execution tool** multiple times. First, it uses the tool to look up each team’s `team_id` in the **teams** table (`SELECT team_id FROM teams WHERE team = ...`). Then, using those IDs, it calls the same tool again to fetch the coach names from the **coaches** table (`SELECT coach_name FROM coaches WHERE team_id = ...`). After the tool returns the data, the LLL uses its internal knowledge to generate short biographical sentences based on the retrieved coach names.

In [ ]:
invoke_agent_claude(f"Considering that {away_team} played against {home_team}, find in the database the names of the coaches and see what you already know about them, and create one sentence each with a mini biography of each one. Return only the two sentences, no intros or questions.")

======> SELECT c.coach_name, t.team FROM coaches c JOIN teams t ON c.team_id = t.team_id WHERE t.team IN ('Army West Point', 'Navy');


"Curt Cignetti is a seasoned college football coach with decades of experience, known for his successful stints at programs like Elon, Indiana (PA), and James Madison, where he built a reputation as an offensive-minded leader and consistent winner at multiple levels of collegiate football.\n\nBrent Brennan is a veteran college football coach who gained recognition during his tenure at San Jose State, where he became the program's all-time wins leader before taking the helm at Navy to continue his journey of building competitive teams with discipline and determination."

Simulating the example of a game result that did not really came from a real event...

In [ ]:

home_team = 'Navy'
away_team = 'Troy'
home_score = 20
away_score = 11

invoke_agent_claude(f"Find the mascots of these teams in the database: {away_team} and {home_team}. Considering that the score was {away_score} x {home_score}, create a message to be posted on social media mentioning the teams' mascots. Return only the post, no intros or questions.")

======> SELECT team, nickname FROM teams WHERE team IN ('Troy', 'Navy');


'⚓🏈 The **Navy Mids** dominated the **Troy Trojans** in a commanding performance, sailing to a convincing **20-11** victory! 💙💛\n\nThe Trojans put up a fight, but the Mids were simply unstoppable today, showing why they rule the seas AND the gridiron! ⚔️🌊\n\nNo stopping this wave! The Mids are ROLLING! 🚢💪\n\n📢 Were you watching? Let us know your thoughts below! 👇\n\n#NavyMids #TroyTrojans #CollegeFootball #NavyWins #GameDay'

... to test the memory capability, in addition to all the tools call.

In [ ]:
team = 'Navy'
invoke_agent_claude(f'Create a short paragraph about {team}, including its players, coach, and nickname. Also, list its most recent scores that you know of. Return only the paragraph, no intros or questions.')

======> SELECT p.player_name, p.number FROM players p JOIN teams t ON p.team_id = t.team_id WHERE t.team = 'Navy';
======> SELECT c.coach_name FROM coaches c JOIN teams t ON c.team_id = t.team_id WHERE t.team = 'Navy';
======> SELECT nickname FROM teams WHERE team = 'Navy';


"The **Navy Mids**, led by head coach **Brent Brennan**, are one of college football's most storied programs, known for their discipline, toughness, and military tradition. With player **Noah Fifita (#91)** suiting up for the Mids, the team continues to bring competitive football to the field. In their most recent outings, Navy secured a thrilling **17-16 victory over Army West Point** in their iconic rivalry game, and also delivered a dominant **20-11 win over Troy**, showcasing the team's ability to grind out wins and control the game on both sides of the ball. The Mids continue to prove that their brand of hard-nosed football is a force to be reckoned with in college football! ⚓🏈"

The **Navy Mids**, led by head coach **Brent Brennan**, are one of college football's most storied programs, known for their discipline, toughness, and military tradition. With player **Noah Fifita (#91)** suiting up for the Mids, the team continues to bring competitive football to the field. In their most recent outings, Navy secured a thrilling **17-16 victory over Army West Point** in their iconic rivalry game, and also delivered a dominant **20-11 win over Troy**, showcasing the team's ability to grind out wins and control the game on both sides of the ball. The Mids continue to prove that their brand of hard-nosed football is a force to be reckoned with in college football! ⚓🏈